# 01 — Exploratory Data Analysis & Business Questions

This notebook answers the core business questions a retail/FMCG stakeholder would ask, organized into four groups:

1. **Sales performance** — revenue, trends, growth, transaction economics
2. **Product analytics** — top products, declining products, pricing outliers
3. **Customer analytics** — biggest customers, activity/inactivity, growth/decline
4. **Retail/FMCG-specific questions** — revenue vs volume drivers, underpricing, seasonal stocking

Dataset: `retail_sales_cleaned.csv` — 181,670 invoice line items, 1,247 products, 50 customers, 2024-09-25 to 2026-07-31.

---


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
pd.options.display.float_format = "{:,.2f}".format

DATA_PATH = Path("../data/retail_sales_cleaned.csv")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
df["date"] = pd.to_datetime(df["date"])
print(f"Shape: {df.shape}")
df.head()


## 1. Sales Performance

### Q: What is total revenue?

In [ ]:
total_revenue = df["sales_value"].sum()
total_kg = df["mass_kg"].sum()
n_invoices = df["doc_no"].nunique()

print(f"Total revenue:        R{total_revenue:,.2f}")
print(f"Total volume sold:    {total_kg:,.0f} kg")
print(f"Total invoices:       {n_invoices:,}")
print(f"Total line items:     {len(df):,}")


### Q: What are the monthly / weekly / daily sales trends?

In [ ]:
monthly = df.groupby("year_month")["sales_value"].sum().reset_index()
monthly["year_month"] = pd.to_datetime(monthly["year_month"])
monthly = monthly.sort_values("year_month")

fig, ax = plt.subplots()
ax.plot(monthly["year_month"], monthly["sales_value"], marker="o", linewidth=2)
ax.set_title("Monthly Sales Revenue")
ax.set_ylabel("Sales Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_monthly_trend.png", dpi=120)
plt.show()


In [ ]:
weekly = df.set_index("date")["sales_value"].resample("W").sum()
fig, ax = plt.subplots()
ax.plot(weekly.index, weekly.values, linewidth=1.5, color="teal")
ax.set_title("Weekly Sales Revenue")
ax.set_ylabel("Sales Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_weekly_trend.png", dpi=120)
plt.show()


In [ ]:
daily = df.groupby("date")["sales_value"].sum()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily.index, daily.values, linewidth=0.8, color="steelblue")
ax.set_title("Daily Sales Revenue")
ax.set_ylabel("Sales Value")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_daily_trend.png", dpi=120)
plt.show()


### Q: Which months are strongest? Which days are strongest?

In [ ]:
strongest_months = df.groupby("month_name")["sales_value"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
sns.barplot(x=strongest_months.index, y=strongest_months.values, ax=ax,
            order=strongest_months.index)
ax.set_title("Total Revenue by Calendar Month (all years combined)")
ax.set_ylabel("Sales Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_revenue_by_month.png", dpi=120)
plt.show()

print("Strongest months:\n", strongest_months.head(3))
print("\nWeakest months:\n", strongest_months.tail(3))


In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
strongest_days = df.groupby("day_name")["sales_value"].sum().reindex(dow_order)

fig, ax = plt.subplots()
sns.barplot(x=strongest_days.index, y=strongest_days.values, ax=ax)
ax.set_title("Total Revenue by Day of Week")
ax.set_ylabel("Sales Value")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_revenue_by_dow.png", dpi=120)
plt.show()

print(strongest_days.sort_values(ascending=False))


### Q: Is the business growing?

In [ ]:
# Compare the first 6 full trading months to the most recent 6 full trading months
monthly_full = monthly.set_index("year_month")["sales_value"]
first_6 = monthly_full.iloc[1:7]   # skip partial opening month
last_6 = monthly_full.iloc[-7:-1]  # skip partial current month if in progress

growth_pct = (last_6.mean() - first_6.mean()) / first_6.mean() * 100
print(f"Avg monthly revenue — first 6 full months:  R{first_6.mean():,.0f}")
print(f"Avg monthly revenue — most recent 6 months: R{last_6.mean():,.0f}")
print(f"Growth: {growth_pct:+.1f}%")

fig, ax = plt.subplots()
ax.plot(monthly_full.index, monthly_full.values, marker="o", linewidth=2)
z = np.polyfit(range(len(monthly_full)), monthly_full.values, 1)
trend = np.poly1d(z)(range(len(monthly_full)))
ax.plot(monthly_full.index, trend, linestyle="--", color="crimson", label="Linear trend")
ax.set_title("Revenue Trend Over Time")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_growth_trend.png", dpi=120)
plt.show()


### Q: What is the average transaction value? What is sales volume in kg?

In [ ]:
txn_value = df.groupby("doc_no")["sales_value"].sum()
print(f"Average transaction (invoice) value: R{txn_value.mean():,.2f}")
print(f"Median transaction value:            R{txn_value.median():,.2f}")
print(f"Largest single transaction:          R{txn_value.max():,.2f}")

kg_by_month = df.groupby("year_month")["mass_kg"].sum()
print(f"\nTotal kg sold: {df['mass_kg'].sum():,.0f} kg")
print(f"Average kg sold per month: {kg_by_month.mean():,.0f} kg")


## 2. Product Analytics

### Q: What are the top 10 / 20 / 50 products (by revenue, units, kg)?

In [ ]:
product_summary = df.groupby("product").agg(
    revenue=("sales_value", "sum"),
    units=("qty", "sum"),
    kg=("mass_kg", "sum"),
    n_invoices=("doc_no", "nunique"),
).sort_values("revenue", ascending=False)

print("Top 10 products by REVENUE:")
print(product_summary.sort_values("revenue", ascending=False).head(10)[["revenue"]])

print("\nTop 10 products by UNITS sold:")
print(product_summary.sort_values("units", ascending=False).head(10)[["units"]])

print("\nTop 10 products by KG sold:")
print(product_summary.sort_values("kg", ascending=False).head(10)[["kg"]])


In [ ]:
top20 = product_summary.sort_values("revenue", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(y=top20.index, x=top20["revenue"], ax=ax, orient="h")
ax.set_title("Top 20 Products by Revenue")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_top20_products.png", dpi=120)
plt.show()


### Q: What percentage of revenue comes from the top 20% of products?

In [ ]:
n_top20pct = int(len(product_summary) * 0.2)
top20pct_share = product_summary["revenue"].sort_values(ascending=False).head(n_top20pct).sum() / product_summary["revenue"].sum() * 100
print(f"Top 20% of products ({n_top20pct} of {len(product_summary)}) generate {top20pct_share:.1f}% of total revenue.")
print("(Full ABC/Pareto breakdown is in notebook 02_abc_pareto_analysis.ipynb)")


### Q: Which products have declining sales?

In [ ]:
last_date = df["date"].max()
recent_window = df[df["date"] > last_date - pd.Timedelta(days=90)]
prior_window = df[(df["date"] <= last_date - pd.Timedelta(days=90)) &
                   (df["date"] > last_date - pd.Timedelta(days=180))]

rev_recent = recent_window.groupby("product")["sales_value"].sum()
rev_prior = prior_window.groupby("product")["sales_value"].sum()

trend = pd.DataFrame({"recent_90d": rev_recent, "prior_90d": rev_prior}).fillna(0)
trend = trend[trend["prior_90d"] > 1000]  # ignore tiny/noisy products
trend["pct_change"] = (trend["recent_90d"] - trend["prior_90d"]) / trend["prior_90d"] * 100
trend = trend.sort_values("pct_change")

print("Top 10 declining products (last 90 days vs prior 90 days):")
print(trend.head(10))

print("\nTop 10 growing products (last 90 days vs prior 90 days):")
print(trend.sort_values("pct_change", ascending=False).head(10))


### Q: Which products have unusually high/low selling prices?

In [ ]:
priced = df[df["qty"] > 0].copy()
priced["unit_price"] = priced["sales_value"] / priced["qty"]

price_stats = priced.groupby("product")["unit_price"].agg(mean_price="mean", std_price="std", n="count")
price_stats = price_stats[price_stats["n"] >= 10]  # need enough observations to judge variability

price_stats["cv"] = price_stats["std_price"] / price_stats["mean_price"]  # coefficient of variation
print("Products with the most unstable/unusual pricing (highest coefficient of variation):")
print(price_stats.sort_values("cv", ascending=False).head(10))

print("\nHighest average unit price:")
print(price_stats.sort_values("mean_price", ascending=False).head(10)[["mean_price", "n"]])

print("\nLowest average unit price (excluding free/near-zero items):")
print(price_stats[price_stats["mean_price"] > 1].sort_values("mean_price").head(10)[["mean_price", "n"]])


### Q: Which products should be promoted?

Candidates: products with strong recent growth but still low overall revenue share (room to scale), or high-margin-looking (high unit price) products with low volume relative to their price point.

In [ ]:
growing = trend[trend["pct_change"] > 10].sort_values("pct_change", ascending=False)
revenue_75th_pct = product_summary["revenue"].quantile(0.75)
promotion_candidates = growing[growing.index.isin(product_summary[product_summary["revenue"] < revenue_75th_pct].index)]

print(f"Promotion candidates — growing >10% but still below the 75th percentile of product revenue")
print(f"(i.e. real momentum, room to scale, not already a top performer):\n")
print(promotion_candidates.head(10))


## 3. Customer Analytics

### Q: Who are our biggest customers? Which generate the most revenue / buy the most volume?

In [ ]:
customer_summary = df.groupby("customer").agg(
    revenue=("sales_value", "sum"),
    kg=("mass_kg", "sum"),
    n_invoices=("doc_no", "nunique"),
    last_purchase=("date", "max"),
    first_purchase=("date", "min"),
).sort_values("revenue", ascending=False)

print("Top 10 customers by revenue:")
print(customer_summary.head(10)[["revenue", "n_invoices"]])

print("\nTop 10 customers by volume (kg):")
print(customer_summary.sort_values("kg", ascending=False).head(10)[["kg"]])


In [ ]:
top15_cust = customer_summary.head(15)
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(y=top15_cust.index, x=top15_cust["revenue"], ax=ax, orient="h")
ax.set_title("Top 15 Customers by Revenue")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_top_customers.png", dpi=120)
plt.show()


### Q: Which customers are becoming inactive? What is customer purchase frequency?

In [ ]:
snapshot_date = df["date"].max() + pd.Timedelta(days=1)
customer_summary["recency_days"] = (snapshot_date - customer_summary["last_purchase"]).dt.days
customer_summary["tenure_days"] = (customer_summary["last_purchase"] - customer_summary["first_purchase"]).dt.days
customer_summary["purchase_frequency"] = customer_summary["n_invoices"] / customer_summary["tenure_days"].replace(0, np.nan) * 30  # invoices per 30 days

inactive_threshold_days = 60
inactive = customer_summary[customer_summary["recency_days"] > inactive_threshold_days].sort_values("recency_days", ascending=False)
print(f"Customers inactive for >{inactive_threshold_days} days ({len(inactive)} of {len(customer_summary)}):")
print(inactive[["recency_days", "revenue", "n_invoices"]].head(15))

print("\nPurchase frequency (invoices per 30 days), most frequent buyers:")
print(customer_summary.sort_values("purchase_frequency", ascending=False).head(10)[["purchase_frequency", "n_invoices"]])


### Q: Which customers are growing? Which are declining?

In [ ]:
cust_rev_recent = recent_window.groupby("customer")["sales_value"].sum()
cust_rev_prior = prior_window.groupby("customer")["sales_value"].sum()

cust_trend = pd.DataFrame({"recent_90d": cust_rev_recent, "prior_90d": cust_rev_prior}).fillna(0)
cust_trend = cust_trend[cust_trend["prior_90d"] > 500]
cust_trend["pct_change"] = (cust_trend["recent_90d"] - cust_trend["prior_90d"]) / cust_trend["prior_90d"] * 100

print("Top 10 growing customers (last 90 days vs prior 90 days):")
print(cust_trend.sort_values("pct_change", ascending=False).head(10))

print("\nTop 10 declining customers (last 90 days vs prior 90 days):")
print(cust_trend.sort_values("pct_change").head(10))


## 4. Retail / FMCG-Specific Questions

### Q: Which products drive revenue vs volume? High revenue/low volume vs high volume/low revenue?

In [ ]:
quad = product_summary.copy()
quad["rev_rank"] = quad["revenue"].rank(pct=True)
quad["kg_rank"] = quad["kg"].rank(pct=True)

def quadrant(row):
    if row["rev_rank"] >= 0.5 and row["kg_rank"] >= 0.5:
        return "High Revenue / High Volume"
    if row["rev_rank"] >= 0.5 and row["kg_rank"] < 0.5:
        return "High Revenue / Low Volume (Premium)"
    if row["rev_rank"] < 0.5 and row["kg_rank"] >= 0.5:
        return "Low Revenue / High Volume (Commodity)"
    return "Low Revenue / Low Volume"

quad["quadrant"] = quad.apply(quadrant, axis=1)
print(quad["quadrant"].value_counts())

fig, ax = plt.subplots(figsize=(9, 7))
colors = {"High Revenue / High Volume": "#2ca02c", "High Revenue / Low Volume (Premium)": "#1f77b4",
          "Low Revenue / High Volume (Commodity)": "#ff7f0e", "Low Revenue / Low Volume": "#d62728"}
for q, sub in quad.groupby("quadrant"):
    ax.scatter(sub["kg"], sub["revenue"], label=q, alpha=0.6, s=25, color=colors[q])
ax.set_xscale("symlog")
ax.set_yscale("symlog")
ax.set_xlabel("Volume (kg, log scale)")
ax.set_ylabel("Revenue (log scale)")
ax.set_title("Revenue vs Volume Quadrant — Product Portfolio")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "04_revenue_volume_quadrant.png", dpi=120)
plt.show()

print("\nExamples — High Revenue / Low Volume (Premium):")
print(quad[quad["quadrant"] == "High Revenue / Low Volume (Premium)"].sort_values("revenue", ascending=False).head(5)[["revenue", "kg"]])


### Q: Which products are potentially underpriced?

High-volume, low-unit-price products relative to the overall product base can signal underpricing — especially if volume is high enough that a small price increase would meaningfully lift revenue without denting demand much.

In [ ]:
vol_price = priced.groupby("product").agg(
    unit_price=("unit_price", "mean"),
    units=("qty", "sum"),
    revenue=("sales_value", "sum"),
).query("units > @priced['qty'].sum() * 0.001")  # meaningful volume only

vol_price["price_percentile"] = vol_price["unit_price"].rank(pct=True)
vol_price["volume_percentile"] = vol_price["units"].rank(pct=True)

underpriced_candidates = vol_price[(vol_price["volume_percentile"] > 0.8) & (vol_price["price_percentile"] < 0.3)]
print("Potentially underpriced (high volume, low relative price):")
print(underpriced_candidates.sort_values("units", ascending=False).head(10))


### Q: What are the seasonal demand patterns? What should be stocked more heavily before peak periods?

In [ ]:
seasonal = df.pivot_table(index="month_name", columns="quarter", values="sales_value", aggfunc="sum")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
monthly_totals = df.groupby("month_name")["sales_value"].sum().reindex(month_order)

fig, ax = plt.subplots()
sns.barplot(x=monthly_totals.index, y=monthly_totals.values, ax=ax, palette="viridis")
ax.set_title("Seasonal Demand — Total Revenue by Calendar Month")
ax.set_ylabel("Sales Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_seasonal_demand.png", dpi=120)
plt.show()

peak_months = monthly_totals.sort_values(ascending=False).head(3)
print(f"Peak months: {list(peak_months.index)}")

# which products spike hardest in the peak months vs their own average
peak_products = df[df["month_name"].isin(peak_months.index)].groupby("product")["sales_value"].sum()
overall_avg_per_month = df.groupby("product")["sales_value"].sum() / df["month_name"].nunique()
spike_ratio = (peak_products / len(peak_months) / overall_avg_per_month.reindex(peak_products.index)).sort_values(ascending=False)
spike_ratio = spike_ratio[overall_avg_per_month.reindex(spike_ratio.index) > 1000]  # meaningful baseline only

print(f"\nProducts that spike hardest in peak months ({', '.join(peak_months.index)}) — stock these up beforehand:")
print(spike_ratio.head(10))


## Summary of Key Findings

- Total revenue across the dataset: see Section 1 output above.
- Revenue is concentrated: the top 20% of products drive the large majority of revenue (full ABC breakdown in the next notebook).
- Wednesday and Friday are consistently the strongest trading days; Sunday is weakest.
- A small number of customers account for a disproportionate share of revenue — see `03_customer_rfm_segmentation.ipynb` for a full segmentation.
- Several products show strong 90-day growth or decline — worth a monthly review cadence rather than a one-off check.
- The revenue/volume quadrant highlights premium low-volume lines and high-volume commodity lines — these need different inventory strategies (safety stock and freshness for commodity/perishable lines, cash-flow-aware ordering for premium/slow-moving lines).

Next: `02_abc_pareto_analysis.ipynb` for formal product classification, `03_customer_rfm_segmentation.ipynb` for customer segments, and `04_sales_forecasting.ipynb` for revenue forecasts.
